# Task 1 — Supabase → Lakeflow Connect → Unity Catalog (Bronze) + CDC Proof

**What we are doing:**
1. Use Databricks Lakeflow Connect to sync `poc_orders` from Supabase into Unity Catalog Bronze
2. Prove CDC — insert a row in Supabase, run pipeline again, show only 1 row synced

**Storage path (Unity Catalog):**
```
amazon_project.bronze.poc_orders
        │         │       │
     Catalog   Schema   Table
```

> **Hive Metastore vs Unity Catalog:**  
> Hive Metastore = legacy, workspace-level, no governance  
> Unity Catalog = modern, account-level, governed, supports lineage + access control  
> We are using Unity Catalog (`amazon_project`) as per the manager's requirement.

---
**Our Supabase tables (confirmed in pgAdmin):**

| Table | Rows | Notes |
|-------|------|-------|
| cart_items | 16,032 | Shopping cart items |
| carts | 8,000 | Cart sessions |
| product_views | 8,000 | Browsing behavior |
| search_logs | 5,000 | Search activity |
| addresses | 1,478 | Delivery addresses |
| users | 1,000 | Customer accounts |
| profiles | 1,000 | Customer profiles |
| product_variants | 407 | Size/color variants |
| **poc_orders** | **250+** | **Main orders table — our POC** |
| products | 200 | Product catalog |
| promotions | 15 | Promotions |
| categories | 12 | Categories |
| sellers | 10 | Sellers |

**poc_orders columns:**
```
order_id, customer_id, order_date, shipping_date,
expected_delivery_date, actual_delivery_date,
shipping_tier_id, supplier_id, order_channel
```

---
## Part 1 — Lakeflow Connect Setup (done once)

### Step 1: Create the PostgreSQL connection
```
Databricks → Catalog → Connections → + Add connection
  Connection name : postgresqlamazon
  Type            : PostgreSQL
  Host            : aws-0-ap-south-1.pooler.supabase.com
  Port            : 5432
  Database        : postgres
  Username        : postgres.isqcnhvlfnjszllicxqi
  Password        : (from credentials)
  SSL             : Require
  → Test connection → Create
```

### Step 2: Create the ingestion pipeline
```
Databricks → Data Ingestion → PostgreSQL (Preview)
  Connection : postgresqlamazon
  Database   : postgres
  Schema     : public
  Tables     : ✅ poc_orders

  Destination (Unity Catalog):
    Catalog : amazon_project   ← Unity Catalog, NOT hive_metastore
    Schema  : bronze
    Table   : auto-created by pipeline

  Sync mode  : Snapshot + Incremental
  → Create Pipeline → Start
```

### First run result
```
Status   : Completed ✅
Duration : 48 seconds
Upserted : 250 rows   ← full snapshot on first run
Catalog  : amazon_project
Schema   : bronze
Type     : Streaming table
```

---
## Part 2 — Rename the table to poc_orders

Lakeflow Connect auto-names the table `hive_metastore` — rename it to `poc_orders` for clarity.

In [ ]:
# ─── Rename the table from hive_metastore → poc_orders ───────────────────────
# Run this once after the first pipeline run
# Skip this cell if the table is already named poc_orders

%sql
ALTER TABLE amazon_project.bronze.hive_metastore
RENAME TO poc_orders;

---
## Part 3 — Verify First Load

In [ ]:
# ─── Check row count after first pipeline run ─────────────────────────────────

%sql
SELECT COUNT(*) AS total_rows
FROM amazon_project.bronze.poc_orders
-- Expected: 250

In [ ]:
# ─── See sample rows ──────────────────────────────────────────────────────────

%sql
SELECT *
FROM amazon_project.bronze.poc_orders
LIMIT 5

In [ ]:
# ─── Confirm this is Unity Catalog — not Hive Metastore ───────────────────────

%sql
DESCRIBE EXTENDED amazon_project.bronze.poc_orders
-- Look for:
--   Catalog Name : amazon_project   ← Unity Catalog ✅
--   Type         : STREAMING_TABLE

---
## Part 4 — CDC Proof

### What is CDC?

```
Full Load (old way):
  Every run → reads ALL rows → 250 rows re-ingested every time
  Expensive. Slow. Does not scale.

CDC — Change Data Capture (Lakeflow Connect way):
  First run  → full snapshot  → 250 rows
  Next run   → only NEW/CHANGED rows → 1 row, 3 rows, whatever changed
  Fast. Efficient. Production-grade.
```

### How to prove CDC — Step by step

#### Step 1: Go to pgAdmin 4 → run this SQL on Supabase

```sql
-- Insert 1 brand new order
INSERT INTO poc_orders (
    order_id, customer_id, order_date, shipping_date,
    expected_delivery_date, actual_delivery_date,
    shipping_tier_id, supplier_id, order_channel
)
VALUES (
    'OR-TEST-001', 'CUST-99999',
    NOW(), NOW() + INTERVAL '1 day', NOW() + INTERVAL '5 days',
    NULL, 'SHP-001', 'SUP-01', 'Online'
);

-- Simulate delivery — update the same row
UPDATE poc_orders
SET actual_delivery_date = NOW()
WHERE order_id = 'OR-TEST-001';
```

#### Step 2: Databricks → Jobs & Pipelines → postgresqlamazon → Start

#### Step 3: Pipeline result
```
Status   : Completed ✅
Duration : 30 seconds   ← faster than 48s — less data!
Upserted : 1            ← ONLY the 1 changed row synced — NOT all 250!
Deleted  : 0
```

> This is the CDC proof.  
> Supabase had 251 rows. Pipeline detected only the 1 new row and synced just that.  
> Bronze now has 251 rows.

In [ ]:
# ─── Verify Round 2: Bronze should have 251 rows now ──────────────────────────

%sql
SELECT COUNT(*) AS total_rows
FROM amazon_project.bronze.poc_orders
-- Expected: 251  (was 250, +1 from the INSERT in pgAdmin)

In [ ]:
# ─── Find the exact row we inserted — prove it landed in Unity Catalog Bronze ──

%sql
SELECT *
FROM amazon_project.bronze.poc_orders
WHERE order_id = 'OR-TEST-001'

-- Expected output:
--   order_id             : OR-TEST-001
--   customer_id          : CUST-99999
--   order_channel        : Online
--   actual_delivery_date : NOT NULL  (we ran the UPDATE in pgAdmin)

---
## Part 5 — The Full CDC Story (teaching script)

```
RUN 1 — Full Snapshot
──────────────────────────────────────────────────────────
Supabase poc_orders : 250 rows
Pipeline ran        : Upserted 250  (first time — full load)
Duration            : 48 seconds
Destination         : amazon_project.bronze.poc_orders  (Unity Catalog)


ACTION — Change made in Supabase via pgAdmin
──────────────────────────────────────────────────────────
INSERT INTO poc_orders → OR-TEST-001   (1 new row added)
UPDATE poc_orders SET actual_delivery_date = NOW()
       WHERE order_id = 'OR-TEST-001'  (row updated immediately)


RUN 2 — Incremental CDC
──────────────────────────────────────────────────────────
Pipeline ran        : Upserted 1    ← only the 1 change!
Duration            : 30 seconds   ← faster!
Bronze now has      : 251 rows
Proof               : OR-TEST-001 in Unity Catalog with actual_delivery_date filled
```

### Hive Metastore vs Unity Catalog — Why the Manager Asked to Change

| | Hive Metastore | Unity Catalog |
|-|---------------|---------------|
| Governance | None | Fine-grained access control |
| Data lineage | Not available | Full lineage tracking |
| Scope | Workspace only | Account-wide (all workspaces) |
| Audit logs | No | Yes |
| Sharing | Manual | Built-in Delta Sharing |
| Production use | Legacy, avoid for new projects | ✅ Recommended |

### CDC — Full Load vs Incremental

| | Full Load every run | CDC (Lakeflow Connect) |
|-|--------------------|-----------------------|
| 250 rows table | reads 250 every run | reads only what changed |
| 10 million rows table | reads 10M every run | reads 5 changed rows |
| Cost | HIGH | LOW |
| Speed | SLOW | FAST |
| Data freshness | Depends on schedule | Near real-time |

---
## Part 6 — Approach B: JDBC Code (fallback)

Use this if Lakeflow Connect is not available on your Databricks tier.

**Pre-requisite:** Install `org.postgresql:postgresql:42.6.0` on cluster via Maven.

In [ ]:
# ─── JDBC Setup ───────────────────────────────────────────────────────────────
# WARNING: Never commit this notebook to GitHub with the real password

storage_account_name = "amazonprojectadls"
container_name       = "amazon-data"
storage_account_key  = "YOUR_STORAGE_ACCOUNT_KEY"  # ← replace

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)

bronze_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/bronze"

SUPABASE_HOST     = "aws-0-ap-south-1.pooler.supabase.com"
SUPABASE_PORT     = "5432"
SUPABASE_DB       = "postgres"
SUPABASE_USER     = "postgres.isqcnhvlfnjszllicxqi"
SUPABASE_PASSWORD = "YOUR_SUPABASE_PASSWORD"  # ← paste here, never save to Git

jdbc_url = f"jdbc:postgresql://{SUPABASE_HOST}:{SUPABASE_PORT}/{SUPABASE_DB}"

conn_props = {
    "user"    : SUPABASE_USER,
    "password": SUPABASE_PASSWORD,
    "driver"  : "org.postgresql.Driver",
    "ssl"     : "true",
    "sslmode" : "require"
}

print("Setup complete!")

In [ ]:
# ─── Read poc_orders via JDBC and save to Bronze Delta (ADLS) ─────────────────

from pyspark.sql.functions import current_timestamp, lit

poc_orders_df = spark.read.jdbc(
    url        = jdbc_url,
    table      = "poc_orders",
    properties = conn_props
)

poc_orders_df = poc_orders_df \
    .withColumn("_ingested_at", current_timestamp()) \
    .withColumn("_source",      lit("supabase_postgresql"))

poc_orders_bronze_path = f"{bronze_path}/poc_orders"

poc_orders_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(poc_orders_bronze_path)

count = spark.read.format("delta").load(poc_orders_bronze_path).count()
print(f"poc_orders in Bronze (ADLS): {count:,} rows")
print(f"Path: {poc_orders_bronze_path}")

In [ ]:
# ─── Bulk ingest all Supabase tables to Bronze (ADLS) ────────────────────────

supabase_tables = [
    "poc_orders",       # 250+ rows
    "users",            # 1000  rows
    "profiles",         # 1000  rows
    "addresses",        # 1478  rows
    "products",         # 200   rows
    "product_variants", # 407   rows
    "categories",       # 12    rows
    "promotions",       # 15    rows
    "sellers",          # 10    rows
    "cart_items",       # 16032 rows
    "carts",            # 8000  rows
    "product_views",    # 8000  rows
    "search_logs",      # 5000  rows
]

print(f"{'Table':<25} {'Rows':>8}  Status")
print("-" * 55)

for table_name in supabase_tables:
    try:
        df = spark.read.jdbc(url=jdbc_url, table=table_name, properties=conn_props)
        df = df \
            .withColumn("_ingested_at", current_timestamp()) \
            .withColumn("_source",      lit("supabase_postgresql"))
        df.write.format("delta").mode("overwrite").save(f"{bronze_path}/{table_name}")
        row_count = spark.read.format("delta").load(f"{bronze_path}/{table_name}").count()
        print(f"{table_name:<25} {row_count:>8,}  Saved to Bronze")
    except Exception as e:
        print(f"{table_name:<25} {'':>8}  ERROR: {str(e)[:60]}")

print("-" * 55)
print("Done!")

---
## Summary

| | Lakeflow Connect | JDBC Code |
|-|-----------------|----------|
| Destination | Unity Catalog (`amazon_project.bronze`) | ADLS Delta (`bronze/poc_orders`) |
| CDC | Built-in automatic | Manual watermark needed |
| First run | Full snapshot (250 rows) | Full overwrite |
| Next runs | Only changed rows (1 row) | Full overwrite unless watermark added |
| Setup | UI — no code | Python code |

**Final Bronze location:**
```
amazon_project.bronze.poc_orders   ← Lakeflow Connect → Unity Catalog ✅
```